In [11]:
# Imports
import os
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import joblib
import random
import pickle
import numpy as np
from statistics import mean, stdev
from sklearn.metrics import roc_auc_score, precision_score, recall_score, confusion_matrix, f1_score
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import RepeatedStratifiedKFold
import config
from preprocessing_utils import *

In [14]:
# --------------------------
# Set seeds for reproducibility
# --------------------------
SEED = 100
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
g = torch.Generator()
g.manual_seed(SEED)


# --------------------------
# Hyperparameters: (for my own reference)
# --------------------------
# lr (learning rate)
# batch size 32
# hidden_dim
# number_of_epochs 
# patience

# =======================
# Custom Dataset
# =======================
# class MultimodalDataset(Dataset):
#     def __init__(self, clin_path, mrna_path, mut_path, labels_path):
#         # Load preprocessed data
#         self.clinical = joblib.load(clin_path).to_numpy().astype(float)
#         self.mrna = joblib.load(mrna_path).to_numpy().astype(float)
#         self.mutation = joblib.load(mut_path).to_numpy().astype(float)
#         self.labels = joblib.load(labels_path).to_numpy().astype(float)
#         print(self.clinical.shape)
#         print(self.mrna.shape)
#         print(self.mutation.shape)
                
#     def __len__(self):
#         return len(self.labels)
    
#     def __getitem__(self, idx):
#         return (
#             torch.tensor(self.clinical[idx], dtype=torch.float32),
#             torch.tensor(self.mrna[idx], dtype=torch.float32),
#             torch.tensor(self.mutation[idx], dtype=torch.float32),
#             torch.tensor(self.labels[idx], dtype=torch.float32)
#         )

# =======================
# Paths
# =======================
# base_dir = "../split_data_balanced"

# train_dataset = MultimodalDataset(
#     f"{base_dir}/train/clinical.pkl",
#     f"{base_dir}/train/mrna.pkl",
#     f"{base_dir}/train/mutation.pkl",
#     f"{base_dir}/train/labels.pkl"
# )
# # FIXME TESTING CHANGED VAL AND TEST
# val_dataset = MultimodalDataset(
#     f"{base_dir}/val/clinical.pkl",
#     f"{base_dir}/val/mrna.pkl",
#     f"{base_dir}/val/mutation.pkl",
#     f"{base_dir}/val/labels.pkl"
# )

# test_dataset = MultimodalDataset(
#     f"{base_dir}/test/clinical.pkl",
#     f"{base_dir}/test/mrna.pkl",
#     f"{base_dir}/test/mutation.pkl",
#     f"{base_dir}/test/labels.pkl"
# )


class MultimodalDataset(Dataset):
    def __init__(self, X, y, clinical_cols, mrna_cols, mutation_cols):
        # Split modalities
        self.clinical = X[clinical_cols].to_numpy()
        self.mrna = X[mrna_cols].to_numpy().astype(float)
        self.mutation = X[mutation_cols].to_numpy().astype(float)
        self.labels = y.to_numpy().astype(float)
                
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return (
            torch.tensor(self.clinical[idx], dtype=torch.float32),
            torch.tensor(self.mrna[idx], dtype=torch.float32),
            torch.tensor(self.mutation[idx], dtype=torch.float32),
            torch.tensor(self.labels[idx], dtype=torch.float32)
        )

X_train = joblib.load(os.path.join(config.SPLIT_DATA_DIR, "X_train.joblib"))
y_train = joblib.load(os.path.join(config.SPLIT_DATA_DIR, "y_train.joblib"))
X_val = joblib.load(os.path.join(config.SPLIT_DATA_DIR, "X_val.joblib"))
y_val = joblib.load(os.path.join(config.SPLIT_DATA_DIR, "y_val.joblib"))
X_test = joblib.load(os.path.join(config.SPLIT_DATA_DIR, "X_test.joblib"))
y_test = joblib.load(os.path.join(config.SPLIT_DATA_DIR, "y_test.joblib"))
clinical_cols = joblib.load(os.path.join(config.SPLIT_DATA_DIR, "clinical_cols.joblib"))
mrna_cols = joblib.load(os.path.join(config.SPLIT_DATA_DIR, "mrna_cols.joblib"))
mutation_cols = joblib.load(os.path.join(config.SPLIT_DATA_DIR, "mutation_cols.joblib"))

train_dataset = MultimodalDataset(X_train, y_train, clinical_cols, mrna_cols, mutation_cols)
val_dataset = MultimodalDataset(X_val, y_val, clinical_cols, mrna_cols, mutation_cols)
test_dataset = MultimodalDataset(X_test, y_test, clinical_cols, mrna_cols, mutation_cols)

train_loader = DataLoader(
    train_dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=True,
    generator=g,
    worker_init_fn=lambda _: np.random.seed(SEED),
    pin_memory=True,  
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config.BATCH_SIZE,
    generator=g,
    worker_init_fn=lambda _: np.random.seed(SEED),
    pin_memory=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=config.BATCH_SIZE,
    generator=g,
    worker_init_fn=lambda _: np.random.seed(SEED),
    pin_memory=True,
)

# =======================
# Simple Neural Network
# =======================
class SimpleMultimodalNet(nn.Module):
    def __init__(self, clin_dim, mrna_dim, mut_dim, hidden_dim=config.HIDDEN_DIM, dropout=config.DROPOUT, lr=config.LEARNING_RATE):
        super().__init__()

        # Separate encoders for each modality
        self.clinical_fc = nn.Sequential(
            nn.Linear(clin_dim, hidden_dim), 
            nn.ReLU(),
            nn.Dropout(dropout)
            )
        self.mrna_fc = nn.Sequential(
            nn.Linear(mrna_dim, hidden_dim), 
            nn.ReLU(),
            nn.Dropout(dropout)
            )
        self.mut_fc = nn.Sequential(
            nn.Linear(mut_dim, hidden_dim), 
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        # Fusion layer
        self.fusion_fc = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),  # Binary output
        )
        
    def forward(self, clin, mrna, mut):
        clin_emb = self.clinical_fc(clin)
        mrna_emb = self.mrna_fc(mrna)
        mut_emb = self.mut_fc(mut)
        
        # Concatenate embeddings
        fused = torch.cat([clin_emb, mrna_emb, mut_emb], dim=1)
        output = self.fusion_fc(fused)
        return output.squeeze()




In [15]:
# =======================
# Initialize Model
# =======================
# Use one batch to get dimensions
sample_batch = next(iter(train_loader))
clin_dim = sample_batch[0].shape[1]
mrna_dim = sample_batch[1].shape[1]
mut_dim = sample_batch[2].shape[1]

# =======================
# Loss and Optimizer
# =======================
# Convert to tensor and move to the same device as model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleMultimodalNet(clin_dim, mrna_dim, mut_dim).to(device)

# Compute pos_weight
num_pos = (train_dataset.labels == 1).sum()
num_neg = (train_dataset.labels == 0).sum()
pos_weight_val = num_neg / num_pos
pos_weight = torch.tensor(pos_weight_val, dtype=torch.float32, device=device)

# Initialize loss
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=config.LEARNING_RATE)

# --------------------------
# Training loop with validation metrics
# --------------------------
best_auroc = 0
counter = 0

for epoch in range(config.NUM_EPOCHS):
    model.train()
    train_losses = []

    for clin, mrna, mut, labels in train_loader:
        clin = clin.to(device, non_blocking=True)
        mrna = mrna.to(device, non_blocking=True)
        mut = mut.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True).float()
        
        optimizer.zero_grad()
        outputs = model(clin, mrna, mut)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

    avg_train_loss = sum(train_losses) / len(train_losses)

    # --- Validation ---
    model.eval()
    val_losses = []
    all_labels = []
    all_preds = []
    
    with torch.no_grad():
        for clin, mrna, mut, labels in val_loader:
            clin = clin.to(device, non_blocking=True)
            mrna = mrna.to(device, non_blocking=True)
            mut = mut.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True).float()
    
            outputs = model(clin, mrna, mut)
            val_loss = criterion(outputs, labels)
            val_losses.append(val_loss.item())
    
            # collect true labels and sigmoid probabilities (on CPU)
            all_labels.append(labels.detach().cpu())
            all_preds.append(torch.sigmoid(outputs).detach().cpu())

    avg_val_loss = sum(val_losses) / len(val_losses)
    all_labels = torch.cat(all_labels)
    all_preds = torch.cat(all_preds)

    # Binarize predictions at 0.5 threshold
    pred_labels = (all_preds >= 0.5).float()

    val_auroc = roc_auc_score(all_labels.numpy(), all_preds.numpy())
    val_precision = precision_score(all_labels.numpy(), pred_labels.numpy(), zero_division=0)
    val_recall = recall_score(all_labels.numpy(), pred_labels.numpy(), zero_division=0)
    val_f1 = f1_score(all_labels.numpy(), pred_labels.numpy(), zero_division=0)

    print(f"Epoch {epoch+1} - "
          f"Train Loss: {avg_train_loss:.4f}, "
          f"Val Loss: {avg_val_loss:.4f}, "
          f"AUROC: {val_auroc:.4f}, "
          f"Precision: {val_precision:.4f}, "
          f"Recall: {val_recall:.4f}, "
          f"F1: {val_f1:.4f}")

    # --- Early stopping based on AUROC ---
    if val_auroc >= best_auroc:
        best_auroc = val_auroc
        counter = 0
        # Optionally save best model
        torch.save(model.state_dict(), "best_model.pth")
    else:
        counter += 1
        if counter >= config.PATIENCE:
            print(f"Early stopping at epoch {epoch+1} with patience of {config.PATIENCE}")
            break



TypeError: can't convert np.ndarray of type numpy.object_. The only supported types are: float64, float32, float16, complex64, complex128, int64, int32, int16, int8, uint64, uint32, uint16, uint8, and bool.

In [16]:
from itertools import product
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import (
    roc_auc_score, precision_score, recall_score, f1_score,
    average_precision_score, confusion_matrix
)
from statistics import mean, stdev
import numpy as np
import torch, random, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

def run_kfold_gridsearch_with_preprocessing(
    clinical_df, mrna_df, mutation_df, labels,
    external_clinical_df, external_mrna_df, external_mutation_df, external_labels,
    param_grid,
    k=5,
    n_repeats=3,
    optimize_metric='f1'
):
    """
    Grid search with repeated stratified K-fold CV, fitting preprocessors within each fold.
    Uses external validation set (transformed per fold) and adds AUPRC metric.
    """

    def train_one_fold(train_loader, val_loader, model, optimizer, criterion, seed):
        torch.manual_seed(seed)
        np.random.seed(seed)
        random.seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

        best_val_auroc, best_state, patience_counter = 0, None, 0

        for epoch in range(config.NUM_EPOCHS):
            model.train()
            for clin, mrna, mut, y in train_loader:
                clin, mrna, mut, y = clin.to(device), mrna.to(device), mut.to(device), y.to(device)
                optimizer.zero_grad()
                outputs = model(clin, mrna, mut)
                loss = criterion(outputs, y)
                loss.backward()
                optimizer.step()

            # --- Validation phase ---
            model.eval()
            all_probs, all_labels = [], []
            with torch.no_grad():
                for clin, mrna, mut, y in val_loader:
                    clin, mrna, mut, y = clin.to(device), mrna.to(device), mut.to(device), y.to(device)
                    probs = torch.sigmoid(model(clin, mrna, mut)).cpu().numpy()
                    all_probs.append(probs)
                    all_labels.append(y.cpu().numpy())

            all_probs = np.concatenate(all_probs)
            all_labels = np.concatenate(all_labels)
            val_auroc = roc_auc_score(all_labels, all_probs)

            if val_auroc > best_val_auroc:
                best_val_auroc = val_auroc
                best_state = model.state_dict()
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= config.PATIENCE:
                    break

        return best_state

    def evaluate_with_threshold(model, loader, threshold):
        model.eval()
        all_probs, all_labels = [], []
        with torch.no_grad():
            for clin, mrna, mut, y in loader:
                clin, mrna, mut, y = clin.to(device), mrna.to(device), mut.to(device), y.to(device)
                probs = torch.sigmoid(model(clin, mrna, mut)).cpu().numpy()
                all_probs.append(probs)
                all_labels.append(y.cpu().numpy())

        all_probs = np.concatenate(all_probs)
        all_labels = np.concatenate(all_labels)
        preds = (all_probs >= threshold).astype(float)

        metrics = {
            'auroc': roc_auc_score(all_labels, all_probs),
            'auprc': average_precision_score(all_labels, all_probs),
            'precision': precision_score(all_labels, preds, zero_division=0),
            'recall': recall_score(all_labels, preds, zero_division=0),
            'f1': f1_score(all_labels, preds, zero_division=0)
        }
        return metrics

    # --- Initialize folds ---
    rskf = RepeatedStratifiedKFold(n_splits=k, n_repeats=n_repeats, random_state=config.SEED)
    indices = np.arange(len(labels))
    param_combinations = list(product(*param_grid.values()))
    best_hyperparams, best_score = None, -1
    results_summary = {}

    for params in param_combinations:
        param_dict = dict(zip(param_grid.keys(), params))
        print(f"\n===== Hyperparams: {param_dict} =====")

        fold_metrics = {'auroc': [], 'auprc': [], 'precision': [], 'recall': [], 'f1': []}
        external_metrics = {'auroc': [], 'auprc': [], 'precision': [], 'recall': [], 'f1': []}

        for fold, (train_idx, val_idx) in enumerate(rskf.split(indices, labels)):
            print(f"\n--- Fold {fold + 1}/{k} ---")

            # Split data
            clin_train, clin_val = clinical_df.iloc[train_idx], clinical_df.iloc[val_idx]
            mrna_train, mrna_val = mrna_df.iloc[train_idx], mrna_df.iloc[val_idx]
            mut_train, mut_val = mutation_df.iloc[train_idx], mutation_df.iloc[val_idx]
            y_train, y_val = labels.iloc[train_idx], labels.iloc[val_idx]

            # === Initialize and fit each preprocessor on training data ===
            clinical_prep = ClinicalPreprocessorWrapper(
                cols_to_remove=config.CLINICAL_COLS_TO_REMOVE,
                categorical_cols=config.CATEGORICAL_COLS,
                max_null_frac=config.CLINICAL_MAX_NULL_FRAC,
                uniform_thresh=config.CLINICAL_UNIFORM_THRESH,
            )
            mrna_prep = MrnaPreprocessorWrapper(
                max_null_frac=config.MAX_NULL_FRAC,
                uniform_thresh=config.UNIFORM_THRESHOLD,
                corr_thresh=config.CORRELATION_THRESHOLD,
                var_thresh=config.VARIANCE_THRESHOLD,
                re_run_pruning=config.RE_RUN_PRUNING,
                literature_genes=config.LITERATURE_GENES,
                correlated_genes_path=config.CORRELATED_GENES_PATH,
                use_stability_selection=config.USE_STABILITY_SELECTION, 
                n_boots=config.N_BOOTS_FPR, # NOTE: might want to experiment with these values, they are set pretty strict right now and I'm not sure that is good for pytorch
                fpr_alpha=config.FPR_ALPHA, #FIXME: put back to config
                stability_threshold=config.STABILITY_THRESHOLD_FPR, # FIXME: put this back to config
                random_state=config.SEED,
            )
            mutation_prep = MutationPreprocessorWrapper(
                max_null_frac=config.MUTATION_MAX_NULL_FRAC,
                uniform_thresh=config.MUTATION_UNIFORM_THRESH,
            )

            clinical_prep.fit(clin_train)
            mrna_prep.fit(mrna_train, y_train)
            mutation_prep.fit(mut_train)

            # === Transform ===
            clin_train = clinical_prep.transform(clin_train)
            clin_val = clinical_prep.transform(clin_val)
            clin_ext = clinical_prep.transform(external_clinical_df.copy())

            mrna_train = mrna_prep.transform(mrna_train)
            mrna_val = mrna_prep.transform(mrna_val)
            mrna_ext = mrna_prep.transform(external_mrna_df.copy())

            mut_train = mutation_prep.transform(mut_train)
            mut_val = mutation_prep.transform(mut_val)
            mut_ext = mutation_prep.transform(external_mutation_df.copy())

            # --- Build datasets and dataloaders ---
            def to_loader(c, m, mu, y, shuffle=False):
                import pandas as pd
                import numpy as np
                import torch
                from torch.utils.data import TensorDataset, DataLoader
            
                def check_numeric(df, name):
                    if isinstance(df, np.ndarray):
                        df = pd.DataFrame(df)
                    non_numeric_cols = []
                    for col in df.columns:
                        if not pd.api.types.is_numeric_dtype(df[col]):
                            non_numeric_cols.append(col)
                    if non_numeric_cols:
                        print(f"WARNING: {name} has non-numeric columns: {non_numeric_cols}")
                        # Print the first few rows of problematic columns
                        print(df[non_numeric_cols].head())
                    return df.to_numpy(dtype=np.float32)
                
                c = check_numeric(c, "Clinical")
                m = check_numeric(m, "mRNA")
                mu = check_numeric(mu, "Mutation")
            
                if isinstance(y, (pd.DataFrame, pd.Series)):
                    y = y.to_numpy(dtype=np.float32).reshape(-1, 1)
                else:
                    y = np.array(y, dtype=np.float32).reshape(-1, 1)
                y = y.squeeze() # converts from [x, 1] to [x] shape
            
                ds = TensorDataset(
                    torch.tensor(c),
                    torch.tensor(m),
                    torch.tensor(mu),
                    torch.tensor(y)
                )
                return DataLoader(ds, batch_size=config.BATCH_SIZE, shuffle=shuffle)

            # ===# --- Stability selection for mRNA ---
            mrna_ss_params = {
                'n_boots': param_dict.get('mrna_n_boots', config.N_BOOTS_FPR),
                'fpr_alpha': param_dict.get('mrna_fpr_alpha', config.FPR_ALPHA),
                'stability_threshold': param_dict.get('mrna_stability_threshold', config.STABILITY_THRESHOLD_FPR),
                'random_state': config.SEED
            }
            mrna_stability_selector = StabilitySelection(**mrna_ss_params)
            mrna_stability_selector.fit(mrna_train, y_train)
            mrna_train = mrna_stability_selector.transform(mrna_train)
            mrna_val = mrna_stability_selector.transform(mrna_val)
            mrna_ext = mrna_stability_selector.transform(mrna_ext)
            
            
            # --- Stability selection for Mutation ---
            mut_ss_params = {
                'n_boots': param_dict.get('mut_n_boots', config.N_BOOTS_FPR),
                'fpr_alpha': param_dict.get('mut_fpr_alpha', config.FPR_ALPHA),
                'stability_threshold': param_dict.get('mut_stability_threshold', config.STABILITY_THRESHOLD_FPR),
                'random_state': config.SEED
            }
            mut_stability_selector = StabilitySelection(**mut_ss_params)
            mut_stability_selector.fit(mut_train, y_train)
            mut_train = mut_stability_selector.transform(mut_train)
            mut_val = mut_stability_selector.transform(mut_val)
            mut_ext = mut_stability_selector.transform(mut_ext)

            train_loader = to_loader(clin_train, mrna_train, mut_train, y_train, shuffle=True)
            val_loader = to_loader(clin_val, mrna_val, mut_val, y_val)
            ext_loader = to_loader(clin_ext, mrna_ext, mut_ext, external_labels)

            # --- Train ---
            model = SimpleMultimodalNet(clin_train.shape[1], mrna_train.shape[1], mut_train.shape[1], param_dict["hidden_dim"], param_dict["dropout"], param_dict["lr"]).to(device)
            optimizer = torch.optim.Adam(model.parameters(), lr=param_dict.get('lr', 1e-3))
            criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

            best_state = train_one_fold(train_loader, val_loader, model, optimizer, criterion, config.SEED + fold)
            model.load_state_dict(best_state)

            # --- Find ideal threshold on validation set ---
            all_probs, all_labels = [], []
            with torch.no_grad():
                for clin, mrna, mut, y in val_loader:
                    clin, mrna, mut, y = clin.to(device), mrna.to(device), mut.to(device), y.to(device)
                    probs = torch.sigmoid(model(clin, mrna, mut)).cpu().numpy()
                    all_probs.append(probs)
                    all_labels.append(y.cpu().numpy())
            all_probs = np.concatenate(all_probs)
            all_labels = np.concatenate(all_labels)

            thresholds = np.linspace(0.001, 0.9, 200)
            best_t, best_metric = 0.5, -1
            for t in thresholds:
                preds = (all_probs >= t).astype(float)
                score = f1_score(all_labels, preds, zero_division=0)
                if score > best_metric:
                    best_metric, best_t = score, t

            # --- Evaluate validation and external sets ---
            val_metrics = evaluate_with_threshold(model, val_loader, best_t)
            ext_metrics = evaluate_with_threshold(model, ext_loader, best_t)

            print(f"Val metrics: {val_metrics}")
            print(f"Ext metrics: {ext_metrics}")

            for k_ in fold_metrics.keys():
                fold_metrics[k_].append(val_metrics[k_])
                external_metrics[k_].append(ext_metrics[k_])

        results_summary[str(param_dict)] = {
            "internal": {k: (mean(v), stdev(v)) for k, v in fold_metrics.items()},
            "external": {k: (mean(v), stdev(v)) for k, v in external_metrics.items()}
        }

        mean_f1 = results_summary[str(param_dict)]["internal"]["f1"][0]
        if mean_f1 > best_score:
            best_score = mean_f1
            best_hyperparams = param_dict

    print(f"\n=== Best hyperparameters: {best_hyperparams} (mean F1 = {best_score:.4f}) ===")
    return results_summary, best_hyperparams


param_grid = {
    'dropout': [0],
    'hidden_dim': [32],
    'lr': [1e-3],
    # mRNA stability selection hyperparams
    'mrna_n_boots': [50],
    'mrna_fpr_alpha': [0.05],
    'mrna_stability_threshold': [0.75],
    # Mutation stability selection hyperparams
    'mut_n_boots': [50],
    'mut_fpr_alpha': [0.05],
    'mut_stability_threshold': [0.75]

}

X_train = joblib.load(os.path.join(config.SPLIT_DATA_DIR, "X_train.joblib"))
y_train = joblib.load(os.path.join(config.SPLIT_DATA_DIR, "y_train.joblib"))
X_val = joblib.load(os.path.join(config.SPLIT_DATA_DIR, "X_val.joblib"))
y_val = joblib.load(os.path.join(config.SPLIT_DATA_DIR, "y_val.joblib"))
X_test = joblib.load(os.path.join(config.SPLIT_DATA_DIR, "X_test.joblib"))
y_test = joblib.load(os.path.join(config.SPLIT_DATA_DIR, "y_test.joblib"))
clinical_cols = joblib.load(os.path.join(config.SPLIT_DATA_DIR, "clinical_cols.joblib"))
mrna_cols = joblib.load(os.path.join(config.SPLIT_DATA_DIR, "mrna_cols.joblib"))
mutation_cols = joblib.load(os.path.join(config.SPLIT_DATA_DIR, "mutation_cols.joblib"))

# Split modalities
clinical_train = X_train[clinical_cols]
mrna_train = X_train[mrna_cols]
mutation_train = X_train[mutation_cols]

clinical_val = X_val[clinical_cols]
mrna_val = X_val[mrna_cols]
mutation_val = X_val[mutation_cols]

clinical_test = X_test[clinical_cols]
mrna_test = X_test[mrna_cols]
mutation_test = X_test[mutation_cols]


results_summary, best_hyperparams = run_kfold_gridsearch_with_preprocessing(
    clinical_train, mrna_train, mutation_train, y_train,
    clinical_val, mrna_val, mutation_val, y_val,
    param_grid,
    k=3,
    n_repeats=1,
    optimize_metric='f1'
)



===== Hyperparams: {'dropout': 0, 'hidden_dim': 32, 'lr': 0.001, 'mrna_n_boots': 50, 'mrna_fpr_alpha': 0.05, 'mrna_stability_threshold': 0.75, 'mut_n_boots': 50, 'mut_fpr_alpha': 0.05, 'mut_stability_threshold': 0.75} =====

--- Fold 1/3 ---


NameError: name 'device' is not defined

In [ ]:
# #### kfold not including preprocessing ####

# from itertools import product
# from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, confusion_matrix
# from statistics import mean, stdev
# from sklearn.model_selection import RepeatedStratifiedKFold
# import numpy as np
# import torch, random, torch.nn as nn
# from torch.utils.data import DataLoader

# def run_kfold_gridsearch_with_external_val(
#     train_dataset,
#     external_val_dataset,
#     param_grid,
#     k=5,
#     n_repeats=3,
#     optimize_metric='f1'
# ):
#     """
#     Performs grid search over hyperparameters with repeated stratified k-fold cross-validation.
#     Prints fold-level metrics and final best hyperparameters.
#     """

#     def train_one_fold(train_loader, val_loader, model, optimizer, criterion, seed):
#         print("Training next fold...")
#         torch.manual_seed(seed)
#         torch.cuda.manual_seed_all(seed)
#         np.random.seed(seed)
#         random.seed(seed)
#         torch.backends.cudnn.deterministic = True
#         torch.backends.cudnn.benchmark = False

#         best_val_auroc, best_state, patience_counter = 0, None, 0

#         for epoch in range(config.NUM_EPOCHS):
#             model.train()
#             for clin, mrna, mut, labels_batch in train_loader:
#                 clin, mrna, mut, labels_batch = (
#                     clin.to(device),
#                     mrna.to(device),
#                     mut.to(device),
#                     labels_batch.to(device),
#                 )
#                 optimizer.zero_grad()
#                 outputs = model(clin, mrna, mut)
#                 loss = criterion(outputs, labels_batch)
#                 loss.backward()
#                 optimizer.step()

#             # Validation
#             model.eval()
#             all_labels, all_probs = [], []
#             with torch.no_grad():
#                 for clin, mrna, mut, labels_batch in val_loader:
#                     clin, mrna, mut, labels_batch = (
#                         clin.to(device),
#                         mrna.to(device),
#                         mut.to(device),
#                         labels_batch.to(device),
#                     )
#                     outputs = model(clin, mrna, mut)
#                     probs = torch.sigmoid(outputs).cpu().numpy()
#                     all_probs.append(probs)
#                     all_labels.append(labels_batch.cpu().numpy())

#             all_labels = np.concatenate(all_labels)
#             all_probs = np.concatenate(all_probs)
#             val_auroc = roc_auc_score(all_labels, all_probs)

#             if val_auroc > best_val_auroc:
#                 best_val_auroc = val_auroc
#                 best_state = model.state_dict()
#                 patience_counter = 0
#             else:
#                 patience_counter += 1
#                 if patience_counter >= config.PATIENCE:
#                     break

#         return best_state

#     labels = train_dataset.labels
#     indices = np.arange(len(labels))
#     ext_val_loader = DataLoader(external_val_dataset, batch_size=config.BATCH_SIZE)

#     rskf = RepeatedStratifiedKFold(n_splits=k, n_repeats=n_repeats, random_state=SEED)
#     param_combinations = list(product(*param_grid.values()))
#     best_hyperparams, best_score = None, -1
#     results_summary = {}

#     for params in param_combinations:
#         param_dict = dict(zip(param_grid.keys(), params))
#         print(f"\n===== Testing hyperparams: {param_dict} =====")
#         fold_metrics = {'auroc': [], 'precision': [], 'recall': [], 'f1': []}
#         external_metrics = {'auroc': [], 'precision': [], 'recall': [], 'f1': []}

#         for fold, (train_idx, val_idx) in enumerate(rskf.split(indices, labels)):
#             print(f"\n--- Fold {fold + 1}/{k} ---")
#             train_subset = torch.utils.data.Subset(train_dataset, train_idx)
#             val_subset = torch.utils.data.Subset(train_dataset, val_idx)
#             train_loader = DataLoader(train_subset, batch_size=config.BATCH_SIZE, shuffle=True)
#             val_loader = DataLoader(val_subset, batch_size=config.BATCH_SIZE)

#             model = SimpleMultimodalNet(clin_dim, mrna_dim, mut_dim, **param_dict).to(device)
#             optimizer = torch.optim.Adam(model.parameters())
#             criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

#             best_state = train_one_fold(train_loader, val_loader, model, optimizer, criterion, SEED + fold)
#             model.load_state_dict(best_state)
#             model.eval()

#             # --- Validation predictions ---
#             all_labels, all_probs = [], []
#             with torch.no_grad():
#                 for clin, mrna, mut, labels_batch in val_loader:
#                     clin, mrna, mut, labels_batch = (
#                         clin.to(device),
#                         mrna.to(device),
#                         mut.to(device),
#                         labels_batch.to(device),
#                     )
#                     outputs = model(clin, mrna, mut)
#                     probs = torch.sigmoid(outputs).cpu().numpy()
#                     all_probs.append(probs)
#                     all_labels.append(labels_batch.cpu().numpy())

#             all_labels = np.concatenate(all_labels)
#             all_probs = np.concatenate(all_probs)

#             # --- Find best threshold ---
#             thresholds = np.linspace(0.001, 0.9, 200)
#             best_t, best_m = 0.5, -1
#             for t in thresholds:
#                 preds = (all_probs >= t).astype(float)
#                 if optimize_metric == 'f1':
#                     m = f1_score(all_labels, preds, zero_division=0)
#                 else:
#                     tn, fp, fn, tp = confusion_matrix(all_labels, preds).ravel()
#                     sens, spec = tp / (tp + fn + 1e-9), tn / (tn + fp + 1e-9)
#                     m = sens + spec - 1
#                 if m > best_m:
#                     best_m, best_t = m, t

#             preds = (all_probs >= best_t).astype(float)
#             auroc = roc_auc_score(all_labels, all_probs)
#             prec = precision_score(all_labels, preds, zero_division=0)
#             rec = recall_score(all_labels, preds, zero_division=0)
#             f1 = f1_score(all_labels, preds, zero_division=0)

#             print(f"Fold {fold + 1} internal val metrics:")
#             print(f"  AUROC={auroc:.4f}, Precision={prec:.4f}, Recall={rec:.4f}, F1={f1:.4f}")

#             for k_, v_ in zip(['auroc', 'precision', 'recall', 'f1'], [auroc, prec, rec, f1]):
#                 fold_metrics[k_].append(v_)

#             # --- External validation ---
#             all_labels, all_probs = [], []
#             with torch.no_grad():
#                 for clin, mrna, mut, labels_batch in ext_val_loader:
#                     clin, mrna, mut, labels_batch = (
#                         clin.to(device),
#                         mrna.to(device),
#                         mut.to(device),
#                         labels_batch.to(device),
#                     )
#                     outputs = model(clin, mrna, mut)
#                     probs = torch.sigmoid(outputs).cpu().numpy()
#                     all_probs.append(probs)
#                     all_labels.append(labels_batch.cpu().numpy())

#             all_labels = np.concatenate(all_labels)
#             all_probs = np.concatenate(all_probs)
#             preds = (all_probs >= best_t).astype(float)
#             auroc = roc_auc_score(all_labels, all_probs)
#             prec = precision_score(all_labels, preds, zero_division=0)
#             rec = recall_score(all_labels, preds, zero_division=0)
#             f1 = f1_score(all_labels, preds, zero_division=0)

#             print(f"  External test metrics:")
#             print(f"  AUROC={auroc:.4f}, Precision={prec:.4f}, Recall={rec:.4f}, F1={f1:.4f}")

#             for k_, v_ in zip(['auroc', 'precision', 'recall', 'f1'], [auroc, prec, rec, f1]):
#                 external_metrics[k_].append(v_)

#         # --- Record results for this hyperparameter set ---
#         results_summary[str(param_dict)] = {
#             "internal": {k: (mean(v), stdev(v)) for k, v in fold_metrics.items()},
#             "external": {k: (mean(v), stdev(v)) for k, v in external_metrics.items()}
#         }

#         # print(f"\nAverage metrics for {param_dict}:")
#         for phase in ["internal", "external"]:
#             # print(f"  {phase.upper()}:")
#             for metric, (m, s) in results_summary[str(param_dict)][phase].items():
#                 # print(f"    {metric}: {m:.4f} ± {s:.4f}")

#         mean_f1 = results_summary[str(param_dict)]["internal"]["f1"][0]
#         if mean_f1 > best_score:
#             best_score = mean_f1
#             best_hyperparams = param_dict

#     print(f"\n=== Best hyperparameters: {best_hyperparams} (F1 = {best_score:.4f}) ===")

#     return results_summary, best_hyperparams

    
# param_grid = {
#     'dropout': [0, 0.2, 0.3, 0.4],
#     'hidden_dim': [16, 32, 64, 128],
#     'lr': [1e-2, 1e-3, 1e-4],
# }

# results_summary, best_hyperparams = run_kfold_gridsearch_with_external_val(
#     train_dataset,
#     val_dataset,
#     param_grid,
#     k=3,
#     n_repeats=1,
#     optimize_metric='f1'
# )


# def print_results_summary(results_summary):
#     """
#     Nicely prints the results summary from the k-fold grid search.
#     """
#     for param_str, metrics in results_summary.items():
#         print("="*60)
#         print(f"Hyperparameters: {param_str}")
#         print("-"*60)
#         for phase in ["internal", "external"]:
#             print(f"{phase.upper()} METRICS:")
#             for metric, (mean_val, std_val) in metrics[phase].items():
#                 print(f"  {metric:10}: {mean_val:.4f} ± {std_val:.4f}")
#         print("="*60 + "\n")


# print_results_summary(results_summary)
# print(best_hyperparams)


In [ ]:
def print_results_summary(results_summary):
    """
    Nicely prints the results summary from the k-fold grid search.
    """
    for param_str, metrics in results_summary.items():
        print("="*60)
        print(f"Hyperparameters: {param_str}")
        print("-"*60)
        for phase in ["internal", "external"]:
            print(f"{phase.upper()} METRICS:")
            for metric, (mean_val, std_val) in metrics[phase].items():
                print(f"  {metric:10}: {mean_val:.4f} ± {std_val:.4f}")
        print("="*60 + "\n")

# print_results_summary(results_summary)
print(best_hyperparams)
# Best hyperparameters: {'dropout': 0, 'hidden_dim': 64, 'lr': 0.001}


In [ ]:
from sklearn.metrics import (
    roc_auc_score, precision_score, recall_score, f1_score, confusion_matrix
)
from statistics import mean, stdev
from sklearn.model_selection import RepeatedStratifiedKFold
import numpy as np
import torch, random, torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.calibration import calibration_curve

def run_kfold_crossval_with_external_val(train_dataset, external_val_dataset, k=5, n_repeats=3, optimize_metric='f1'):
    rskf = RepeatedStratifiedKFold(n_splits=k, n_repeats=n_repeats, random_state=SEED)
    labels = train_dataset.labels
    indices = np.arange(len(labels))
    
    fold_aurocs, fold_precisions, fold_recalls, fold_f1s, fold_thresholds = [], [], [], [], []
    external_aurocs, external_precisions, external_recalls, external_f1s = [], [], [], []

    ext_val_loader = DataLoader(external_val_dataset, batch_size=config.BATCH_SIZE)

    for fold, (train_idx, val_idx) in enumerate(rskf.split(indices, labels)):
        print(f"\n===== Fold {fold+1}/{k*n_repeats} =====")

        train_subset = torch.utils.data.Subset(train_dataset, train_idx)
        val_subset = torch.utils.data.Subset(train_dataset, val_idx)

        train_loader = DataLoader(train_subset, batch_size=config.BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(val_subset, batch_size=config.BATCH_SIZE)

        # Reset seeds for reproducibility
        seed = SEED + fold
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

        # Build model
        model = SimpleMultimodalNet(clin_dim, mrna_dim, mut_dim).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=config.LEARNING_RATE, weight_decay=1e-5)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

        best_val_auroc = 0
        best_model_state = None
        counter = 0

        # --- Training with early stopping ---
        for epoch in range(config.NUM_EPOCHS):
            model.train()
            for clin, mrna, mut, labels_batch in train_loader:
                clin, mrna, mut, labels_batch = (
                    clin.to(device),
                    mrna.to(device),
                    mut.to(device),
                    labels_batch.to(device),
                )
                optimizer.zero_grad()
                outputs = model(clin, mrna, mut)
                loss = criterion(outputs, labels_batch)
                loss.backward()
                optimizer.step()

            # --- Validation ---
            model.eval()
            all_labels, all_preds = [], []
            with torch.no_grad():
                for clin, mrna, mut, labels_batch in val_loader:
                    clin, mrna, mut, labels_batch = (
                        clin.to(device),
                        mrna.to(device),
                        mut.to(device),
                        labels_batch.to(device),
                    )
                    outputs = model(clin, mrna, mut)
                    all_labels.append(labels_batch.cpu())
                    all_preds.append(torch.sigmoid(outputs).cpu())

            all_labels = torch.cat(all_labels)
            all_preds = torch.cat(all_preds)
            val_auroc = roc_auc_score(all_labels.numpy(), all_preds.numpy())

            if val_auroc > best_val_auroc:
                best_val_auroc = val_auroc
                best_model_state = model.state_dict()
                counter = 0
            else:
                counter += 1
                if counter >= config.PATIENCE:
                    break

        # --- Evaluate best model on its validation fold ---
        model.load_state_dict(best_model_state)
        model.eval()
        all_labels, all_preds = [], []
        with torch.no_grad():
            for clin, mrna, mut, labels_batch in val_loader:
                clin, mrna, mut, labels_batch = (
                    clin.to(device),
                    mrna.to(device),
                    mut.to(device),
                    labels_batch.to(device),
                )
                outputs = model(clin, mrna, mut)
                all_labels.append(labels_batch.cpu())
                all_preds.append(torch.sigmoid(outputs).cpu())

        all_labels = torch.cat(all_labels).numpy()
        all_preds = torch.cat(all_preds).numpy()

        # --- Choose threshold ---
        best_thresh, best_metric = 0.5, -1
        for t in np.linspace(0.001, 0.9, 200):
            pred_labels = (all_preds >= t).astype(float)
            if optimize_metric == 'f1':
                metric_val = f1_score(all_labels, pred_labels, zero_division=0)
            elif optimize_metric == 'youden':
                tn, fp, fn, tp = confusion_matrix(all_labels, pred_labels).ravel()
                sensitivity = tp / (tp + fn + 1e-9)
                specificity = tn / (tn + fp + 1e-9)
                metric_val = sensitivity + specificity - 1
            if metric_val > best_metric:
                best_metric, best_thresh = metric_val, t

        threshold = best_thresh
        fold_thresholds.append(threshold)
        print(f"Selected optimal threshold={threshold:.2f} ({optimize_metric}={best_metric:.4f})")

        # --- Evaluate on *fold validation* set ---
        pred_labels = (all_preds >= threshold).astype(float)
        auroc = roc_auc_score(all_labels, all_preds)
        precision = precision_score(all_labels, pred_labels, zero_division=0)
        recall = recall_score(all_labels, pred_labels, zero_division=0)
        f1 = f1_score(all_labels, pred_labels, zero_division=0)
        fold_aurocs.append(auroc)
        fold_precisions.append(precision)
        fold_recalls.append(recall)
        fold_f1s.append(f1)

        print(f"Fold {fold+1} AUROC: {auroc:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}")

        # --- Evaluate best model on *external validation set* ---
        all_labels, all_preds = [], []
        with torch.no_grad():
            for clin, mrna, mut, labels_batch in ext_val_loader:
                clin, mrna, mut, labels_batch = (
                    clin.to(device),
                    mrna.to(device),
                    mut.to(device),
                    labels_batch.to(device),
                )
                outputs = model(clin, mrna, mut)
                all_labels.append(labels_batch.cpu())
                all_preds.append(torch.sigmoid(outputs).cpu())

        all_labels = torch.cat(all_labels).numpy()
        all_preds = torch.cat(all_preds).numpy()
        pred_labels = (all_preds >= threshold).astype(float)

        ext_auroc = roc_auc_score(all_labels, all_preds)
        ext_precision = precision_score(all_labels, pred_labels, zero_division=0)
        ext_recall = recall_score(all_labels, pred_labels, zero_division=0)
        ext_f1 = f1_score(all_labels, pred_labels, zero_division=0)
        external_aurocs.append(ext_auroc)
        external_precisions.append(ext_precision)
        external_recalls.append(ext_recall)
        external_f1s.append(ext_f1)

        print(f"→ External Validation AUROC: {ext_auroc:.4f}, Precision: {ext_precision:.4f}, Recall: {ext_recall:.4f}, F1: {ext_f1:.4f}")

    # --- Summary ---
    print("\n==========================")
    print(" K-Fold Summary (Internal) ")
    print("==========================")
    print(f"Mean AUROC    : {mean(fold_aurocs):.4f} ± {stdev(fold_aurocs):.4f}")
    print(f"Mean Precision: {mean(fold_precisions):.4f} ± {stdev(fold_precisions):.4f}")
    print(f"Mean Recall   : {mean(fold_recalls):.4f} ± {stdev(fold_recalls):.4f}")
    print(f"Mean F1-score : {mean(fold_f1s):.4f} ± {stdev(fold_f1s):.4f}")

    print("\n==========================")
    print(" External Validation Summary ")
    print("==========================")
    print(f"Mean AUROC    : {mean(external_aurocs):.4f} ± {stdev(external_aurocs):.4f}")
    print(f"Mean Precision: {mean(external_precisions):.4f} ± {stdev(external_precisions):.4f}")
    print(f"Mean Recall   : {mean(external_recalls):.4f} ± {stdev(external_recalls):.4f}")
    print(f"Mean F1-score : {mean(external_f1s):.4f} ± {stdev(external_f1s):.4f}")


test_dataset = MultimodalDataset(
    f"{base_dir}/test/clinical.pkl",
    f"{base_dir}/test/mrna.pkl",
    f"{base_dir}/test/mutation.pkl",
    f"{base_dir}/test/labels.pkl"
)

test_loader = DataLoader(test_dataset, batch_size=config.BATCH_SIZE)

run_kfold_crossval_with_external_val(train_dataset, test_dataset,  k=5, n_repeats=3, optimize_metric='f1')

With dropout = 0:
Validation AUROC: Mean=0.7799, Std=0.0259
Test AUROC      : Mean=0.7363, Std=0.0236
All Validation Runs: [0.8126, 0.7972, 0.772, 0.772, 0.7455]
All Test Runs       : [0.7083, 0.7664, 0.7411, 0.7173, 0.7485]

In [ ]:
import torch
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_score, recall_score, accuracy_score

# --- Device setup ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# --- Load model ---
model.load_state_dict(torch.load("best_model.pth", map_location=device))
model.to(device)
model.eval()

def evaluate_with_threshold(dataloader, threshold, model, device):
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for batch in dataloader:
            inputs, labels = batch
            inputs = inputs.to(device)
            labels = labels.to(device)

            logits = model(inputs)
            probs = torch.sigmoid(logits).squeeze().detach().cpu()
            labels = labels.detach().cpu()

            all_probs.extend(probs.tolist())
            all_labels.extend(labels.tolist())

    all_probs_tensor = torch.tensor(all_probs)
    all_labels_tensor = torch.tensor(all_labels)

    preds = (all_probs_tensor >= threshold).int()

    auroc = roc_auc_score(all_labels_tensor, all_probs_tensor)
    auprc = average_precision_score(all_labels_tensor, all_probs_tensor)
    f1 = f1_score(all_labels_tensor, preds)
    precision = precision_score(all_labels_tensor, preds)
    recall = recall_score(all_labels_tensor, preds)
    accuracy = accuracy_score(all_labels_tensor, preds)

    return {
        "AUROC": auroc,
        "AUPRC": auprc,
        "F1": f1,
        "Precision": precision,
        "Recall": recall,
        "Accuracy": accuracy
    }

def find_best_threshold(val_loader, model, device):
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for batch in val_loader:
            inputs, labels = batch
            inputs = inputs.to(device)
            labels = labels.to(device)

            logits = model(inputs)
            probs = torch.sigmoid(logits).squeeze().detach().cpu()
            labels = labels.detach().cpu()

            all_probs.extend(probs.tolist())
            all_labels.extend(labels.tolist())

    all_probs_tensor = torch.tensor(all_probs)
    all_labels_tensor = torch.tensor(all_labels)

    thresholds = torch.linspace(0, 1, 101)
    best_f1 = 0.0
    best_thresh = 0.5

    for t in thresholds:
        preds = (all_probs_tensor >= t).int()
        f1 = f1_score(all_labels_tensor, preds)
        if f1 > best_f1:
            best_f1 = f1
            best_thresh = t.item()

    return best_thresh, best_f1

# --- Find best threshold on validation set ---
best_threshold, best_f1 = find_best_threshold(val_loader, model, device)
print(f"\nBest validation F1 threshold: {best_threshold:.3f} (F1 = {best_f1:.3f})")

# --- Evaluate validation metrics ---
val_metrics = evaluate_with_threshold(val_loader, best_threshold, model, device)
print("\nValidation metrics:")
for k, v in val_metrics.items():
    print(f"{k}: {v:.4f}")

# --- Evaluate test metrics ---
test_metrics = evaluate_with_threshold(test_loader, best_threshold, model, device)
print("\nTest metrics (using same threshold):")
for k, v in test_metrics.items():
    print(f"{k}: {v:.4f}")


In [ ]:
state_dict = torch.load("best_model.pth")
model.load_state_dict(state_dict)

# Check the device of model parameters
first_param = next(model.parameters())
print("Model weights are currently on:", first_param.device)
print(device)

model = model.to(device)
model.eval()
all_labels = []
all_probs = []

with torch.no_grad():
    for clin, mrna, mut, labels in val_loader:
        clin = clin.to(device, non_blocking=True)
        mrna = mrna.to(device, non_blocking=True)
        mut = mut.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True).float()

        outputs = model(clin, mrna, mut)
        probs = torch.sigmoid(outputs)
        
        all_labels.append(labels)
        all_probs.append(probs)

# Concatenate batches
all_probs = torch.cat(all_probs).cpu().numpy()
all_labels = torch.cat(all_labels).cpu().numpy()

# --- Find optimal threshold based on F1 score ---
thresholds = np.linspace(0, 1, 101)  # 0.00, 0.01, ..., 1.00
f1_scores = []

for t in thresholds:
    preds = (all_probs >= t).astype(float)
    f1 = f1_score(all_labels, preds)
    f1_scores.append(f1)

best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]
print(f"Optimal threshold for F1: {best_threshold:.2f}, F1 score: {f1_scores[best_idx]:.4f}")

# Use optimal threshold for predictions
preds = (all_probs >= best_threshold).astype(float)

criterion_bce = nn.BCELoss()


# Metrics
val_loss = np.mean(-all_labels * np.log(all_probs + 1e-8) - (1 - all_labels) * np.log(1 - all_probs + 1e-8))
precision = precision_score(all_labels, preds)
recall = recall_score(all_labels, preds)
auroc = roc_auc_score(all_labels, all_probs)
auprc = average_precision_score(all_labels, all_probs)
cm = confusion_matrix(all_labels, preds)

# Unpack confusion matrix
tn, fp, fn, tp = cm.ravel()

print(f"Validation Loss: {val_loss:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"AUROC: {auroc:.4f}")
print(f"AUPRC: {auprc:.4f}")
print("Confusion Matrix:")
print(f"True Negatives: {tn}")
print(f"False Positives: {fp}")
print(f"False Negatives: {fn}")
print(f"True Positives: {tp}")


# ---------------------------
# TEST EVALUATION (use best_threshold found on validation)
# ---------------------------

# Ensure model is in eval mode (you already set it, but safe to repeat)
model.eval()

test_labels_list = []
test_probs_list = []

with torch.no_grad():
    for clin, mrna, mut, labels in test_loader:
        # move inputs to device (same as during val)
        clin = clin.to(device, non_blocking=True)
        mrna = mrna.to(device, non_blocking=True)
        mut = mut.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True).float()

        outputs = model(clin, mrna, mut)          # logits on device
        probs = torch.sigmoid(outputs)             # probabilities on device

        # collect on CPU
        test_probs_list.append(probs.detach().cpu())
        test_labels_list.append(labels.detach().cpu())

# Concatenate and convert to numpy
test_probs = torch.cat(test_probs_list).numpy()
test_labels = torch.cat(test_labels_list).numpy()

# predictions with the selected threshold
test_preds = (test_probs >= best_threshold).astype(float)

# Compute metrics (AUPRC included), using stable numeric ops
from sklearn.metrics import average_precision_score, roc_auc_score, precision_score, recall_score, f1_score, confusion_matrix

# AUPRC
test_auprc = average_precision_score(test_labels, test_probs)

# AUROC
test_auroc = roc_auc_score(test_labels, test_probs)

# Precision / Recall / F1 at threshold
test_precision = precision_score(test_labels, test_preds, zero_division=0)
test_recall = recall_score(test_labels, test_preds, zero_division=0)
test_f1 = f1_score(test_labels, test_preds, zero_division=0)

# Confusion matrix (tn, fp, fn, tp)
cm = confusion_matrix(test_labels, test_preds)
if cm.size == 4:
    tn, fp, fn, tp = cm.ravel()
else:
    # handle edge cases where only one class present
    tn = fp = fn = tp = 0
    if test_labels.sum() == 0:
        tn = cm.ravel()[0]
    else:
        tp = cm.ravel()[0]

# Test loss (same style as validation: numeric BCElike on probabilities)
eps = 1e-8
test_loss = np.mean(-test_labels * np.log(np.clip(test_probs, eps, 1 - eps)) -
                    (1 - test_labels) * np.log(np.clip(1 - test_probs, eps, 1 - eps)))

# Print test results
print("\n--- Test metrics (using best_threshold from validation) ---")
print(f"Threshold: {best_threshold:.3f}")
print(f"Test Loss (BCE on probs): {test_loss:.6f}")
print(f"AUPRC: {test_auprc:.6f}")
print(f"AUROC: {test_auroc:.6f}")
print(f"Precision: {test_precision:.6f}")
print(f"Recall: {test_recall:.6f}")
print(f"F1: {test_f1:.6f}")
print("Confusion Matrix:")
print(f"  TN: {tn}, FP: {fp}, FN: {fn}, TP: {tp}")


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score

def plot_metrics(y_true, y_scores):
    y_true = y_true.cpu().numpy()
    y_scores = y_scores..cpu().numpy()
    
    # ---------- AUROC ----------
    fpr, tpr, _ = roc_curve(y_true, y_scores)
    roc_auc = auc(fpr, tpr)
    
    plt.figure(figsize=(12,5))
    
    plt.subplot(1, 2, 1)
    plt.plot(fpr, tpr, color='blue', label=f'AUROC = {roc_auc:.3f}')
    plt.plot([0,1], [0,1], color='gray', linestyle='--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve')
    plt.legend(loc='lower right')
    
    # ----------  AUPRC ----------
    precision, recall, _ = precision_recall_curve(y_true, y_scores)
    ap_score = average_precision_score(y_true, y_scores)
    
    plt.subplot(1, 2, 2)
    plt.plot(recall, precision, color='green', label=f'AUPRC = {ap_score:.3f}')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title('Precision-Recall Curve')
    plt.legend(loc='lower left')
    
    plt.tight_layout()
    plt.show()

model.eval()
all_labels = []
all_outputs = []
with torch.no_grad():
    for clin, mrna, mut, labels in val_loader:
        clin = clin.to(device, non_blocking=True)
        mrna = mrna.to(device, non_blocking=True)
        mut = mut.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True).float()
        
        outputs = model(clin, mrna, mut)
        all_outputs.append(torch.sigmoid(outputs))
        all_labels.append(labels)

all_labels = torch.cat(all_labels)
all_outputs = torch.cat(all_outputs)

plot_metrics(all_labels, all_outputs)

In [ ]:
model.load_state_dict(torch.load("best_model.pth"))
model.eval()
all_labels = []
all_probs = []

with torch.no_grad():
    for clin, mrna, mut, labels in test_loader:
        clin = clin.to(device, non_blocking=True)
        mrna = mrna.to(device, non_blocking=True)
        mut = mut.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True).float()

        outputs = model(clin, mrna, mut)
        probs = torch.sigmoid(outputs)
        
        all_labels.append(labels)
        all_probs.append(probs)

# Concatenate batches
all_labels = torch.cat(all_labels).cpu().numpy()
all_probs = torch.cat(all_probs).cpu().numpy()

# --- Find optimal threshold based on F1 score ---
thresholds = np.linspace(0, 1, 101)  # 0.00, 0.01, ..., 1.00
f1_scores = []

for t in thresholds:
    preds = (all_probs >= t).astype(float)
    f1 = f1_score(all_labels, preds)
    f1_scores.append(f1)

best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]
print(f"Optimal threshold for F1: {best_threshold:.2f}, F1 score: {f1_scores[best_idx]:.4f}")

# Use optimal threshold for predictions
preds = (all_probs >= best_threshold).astype(float)

# Metrics
test_loss = criterion(torch.tensor(all_probs), torch.tensor(all_labels)).item()
precision = precision_score(all_labels, preds)
recall = recall_score(all_labels, preds)
auroc = roc_auc_score(all_labels, all_probs)
cm = confusion_matrix(all_labels, preds)

# Unpack confusion matrix
tn, fp, fn, tp = cm.ravel()

print(f"Test Loss: {test_loss:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"AUROC: {auroc:.4f}")
print("Confusion Matrix:")
print(f"True Negatives: {tn}")
print(f"False Positives: {fp}")
print(f"False Negatives: {fn}")
print(f"True Positives: {tp}")